# Notebook 3: Your First API Calls
**Time: 45-60 minutes**

### What you'll learn
- Understand what an API is and why it's useful
- Make your first GET request
- Read and interpret responses
- Understand URLs, endpoints, and status codes

---
## 3.1 What is an API?

Think of an API like ordering at a restaurant:

- **You (client)** sit at a table
- **The menu** tells you what's available (API documentation)
- **You tell the waiter** what you want (request)
- **The kitchen** prepares it (server)
- **The waiter brings it back** (response)

You don't go into the kitchen. You don't need to know how they cook. You just need to know how to order.

An API works the same way: you send a request for specific data, and the server sends back a response.

| API is NOT | API IS |
|------------|--------|
| Direct database access | A controlled interface to data |
| A file you download once | A service you can query repeatedly |
| A website for humans to browse | A data endpoint for programs |

---
## 3.2 URLs as addresses

Every API endpoint has a URL:

```
https://restcountries.com/v3.1/name/netherlands
└────────────────────────┘ └────────────────┘
       base URL              endpoint
```

**Query parameters** let you filter or customize the request:

```
https://restcountries.com/v3.1/all?fields=name,capital
                                   └────────────────┘
                                    query parameters
```

This says: "Give me all countries, but only their name and capital."

---
## 3.3 Your first API call

Let's call a real API. We'll use the **REST Countries** API, which provides data about every country in the world.

`REST` means **Representational State Transfer**.
For you, this mostly means: data is exposed through URLs, and you use HTTP methods like `GET` to ask for it.


In [10]:
import requests

response = requests.get("https://restcountries.com/v3.1/name/netherlands")

print(f"Status code: {response.status_code}")  # 200 means success!

Status code: 200


That's it. You just called an API.

- `requests.get()` sends a request to the URL
- `response.status_code` tells you if it worked (`200` = success)
- `response.json()` (next section) converts JSON text into Python data structures


---
## 3.4 Getting the data out

The response contains JSON data. Convert it to Python with `.json()` — this gives you the dictionaries and lists you already know!

In [11]:
data = response.json()
print(f"Type: {type(data)}")   # It's a list!
print(f"Items: {len(data)}")   # How many results?

Type: <class 'list'>
Items: 2


In [12]:
country = data[0]  # Get the first (and only) result

print(country["name"]["common"])    # Netherlands
print(country["capital"][0])        # Amsterdam
print(country["population"])        # Population number

Netherlands
Amsterdam
18100436


This is exactly the nested dictionary navigation you practiced in Notebook 2.

**Exercise:** Call the API for 3 countries and build:
- `country_queries`: a list of 3 country names (lowercase strings)
- `country_summaries`: a list of dictionaries with keys:
  - `name`
  - `capital`
  - `population`

Print `country_summaries` at the end.


In [7]:
# YOUR CODE HERE


[{'tld': ['.iq'], 'cca2': 'IQ', 'ccn3': '368', 'cca3': 'IRQ', 'cioc': 'IRQ', 'independent': True, 'status': 'officially-assigned', 'unMember': True, 'idd': {'root': '+9', 'suffixes': ['64']}, 'capital': ['Baghdad'], 'altSpellings': ['IQ', 'Republic of Iraq', 'Jumhūriyyat al-‘Irāq'], 'region': 'Asia', 'subregion': 'Western Asia', 'landlocked': False, 'borders': ['IRN', 'JOR', 'KWT', 'SAU', 'SYR', 'TUR'], 'area': 438317.0, 'maps': {'googleMaps': 'https://goo.gl/maps/iL8Bmy1sUCW9fUk18', 'openStreetMaps': 'https://www.openstreetmap.org/relation/304934'}, 'population': 46118793, 'fifa': 'IRQ', 'car': {'signs': ['IRQ'], 'side': 'right'}, 'timezones': ['UTC+03:00'], 'continents': ['Asia'], 'flag': '🇮🇶', 'name': {'common': 'Iraq', 'official': 'Republic of Iraq', 'nativeName': {'ara': {'official': 'جمهورية العراق', 'common': 'العراق'}, 'arc': {'official': 'ܩܘܼܛܢܵܐ ܐܝܼܪܲܩ', 'common': 'ܩܘܼܛܢܵܐ'}, 'ckb': {'official': 'کۆماری عێراق', 'common': 'کۆماری'}}}, 'currencies': {'IQD': {'symbol': 'ع.د', 'n

---
## 3.5 Status codes: did it work?

Every response includes a status code that tells you what happened:

| Code | Meaning | What to do |
|------|---------|------------|
| **200** | OK, here's your data | Everything worked |
| **400** | Bad request | Check your parameters |
| **401** | Unauthorized | You need to authenticate |
| **404** | Not found | Check the URL/path |
| **429** | Too many requests | Slow down, wait a bit |
| **500** | Server error | Not your fault, try later |

When debugging, start with the status code:
- `4xx`: usually something in your request
- `5xx`: usually server-side issue

Note: some APIs ignore unknown query parameters instead of failing.
So use intentionally wrong paths/names when you want guaranteed error examples.


In [13]:
test_urls = {
    "valid_country": "https://restcountries.com/v3.1/name/netherlands",
    "misspelled_country": "https://restcountries.com/v3.1/name/netherlnds",
    "wrong_endpoint": "https://restcountries.com/v3.1/nam/netherlands",
}

for label, url in test_urls.items():
    response = requests.get(url)
    print(f"{label}: {response.status_code}")
    if response.status_code != 200:
        print(f"  Response preview: {response.text[:120]}")


valid_country: 200
misspelled_country: 404
  Response preview: {"message":"Not Found","status":404}
bad_fields_query: 200


**Exercise:** Create a dictionary called `status_results` with these keys:
- `valid_country`
- `misspelled_country`
- `wrong_endpoint`

Each value should be the status code from calling the matching URL.

Then create `debug_summary`, a short list of notes that explains what to check for at least two non-200 status codes.


In [ ]:
# YOUR CODE HERE


---
## 3.6 HTTP methods

Every request has a method that says what you want to do:

| Method | Meaning | Analogy |
|--------|---------|----------|
| **GET** | Give me data | Reading a menu |
| **POST** | Here's new data | Placing an order |
| **PUT/PATCH** | Update this | Changing your order |
| **DELETE** | Remove this | Cancelling your order |

A request can also include extra parts:
- `params`: query parameters (filters/options in the URL)
- `headers`: metadata (for example auth token, accepted format)
- `body`: data you send with write requests like POST

We'll keep using GET for now, but this example shows `params` and `headers` too.


In [14]:
# GET request: retrieve a capital (with params + headers)
base_url = "https://restcountries.com/v3.1/name/portugal"
params = {"fields": "name,capital"}
headers = {"Accept": "application/json"}

country_response = requests.get(base_url, params=params, headers=headers)
country_data = country_response.json()[0]
get_capital = country_data["capital"][0]

print(f"Requested URL: {country_response.url}")
print(f"GET status: {country_response.status_code} | Capital: {get_capital}")

# POST request to a read endpoint (expected to fail)
post_response = requests.post(
    base_url,
    json={"capital": "Lisbon"},
    headers=headers,
)
post_status = post_response.status_code
print(f"POST status: {post_status}")

if post_status in [401, 403]:
    print("Write operations often require authentication (API key/token).")
elif post_status in [404, 405]:
    print("This endpoint is read-only or does not allow POST.")
else:
    print("Non-200 means the API rejected the write request.")


GET status: 200 | Capital: Lisbon
POST status: 405
This endpoint is read-only or does not allow POST.


---
## 3.7 Exploring the response

When you get data from an API, it's useful to explore what's in it. Let's look at what the REST Countries API gives us.

In [15]:
response = requests.get("https://restcountries.com/v3.1/name/netherlands")
country = response.json()[0]

# What keys are available?
print("Available fields:")
for key in country.keys():
    print(f"  {key}")

Available fields:
  tld
  cca2
  ccn3
  cca3
  cioc
  independent
  status
  unMember
  idd
  capital
  altSpellings
  region
  subregion
  landlocked
  borders
  area
  maps
  population
  fifa
  car
  timezones
  continents
  flag
  name
  currencies
  languages
  latlng
  demonyms
  translations
  gini
  flags
  coatOfArms
  startOfWeek
  capitalInfo
  postalCode


In [16]:
# Let's look at some interesting fields
print(f"Country: {country['name']['common']}")
print(f"Official name: {country['name']['official']}")
print(f"Capital: {country['capital'][0]}")
print(f"Population: {country['population']}")
print(f"Region: {country['region']}")
print(f"Subregion: {country['subregion']}")
print(f"Area: {country['area']} km\u00b2")
print(f"Borders: {country.get('borders', 'None')}")

Country: Netherlands
Official name: Kingdom of the Netherlands
Capital: Amsterdam
Population: 18100436
Region: Europe
Subregion: Western Europe
Area: 41865.0 km²
Borders: ['BEL', 'DEU']


**Exercise:** Pick any other field from the available fields list and save it as:
- `extra_field_name` (string)
- `extra_field_value` (the value from `country`)

Print both variables.


In [ ]:
# YOUR CODE HERE


---
## 3.8 Checkpoint

Run this cell to verify your understanding. If all checks pass, you're ready to move on!


In [ ]:
# Run this cell to check your understanding

required_vars = [
    "country_queries",
    "country_summaries",
    "status_results",
    "debug_summary",
    "get_capital",
    "post_status",
    "extra_field_name",
    "extra_field_value",
]

missing = [name for name in required_vars if name not in globals()]
assert not missing, (
    "You're missing variables from the exercises: " + ", ".join(missing) +
    ". Complete the exercise cells first, then run this checkpoint again."
)

assert isinstance(country_queries, list), "`country_queries` should be a list"
assert len(country_queries) == 3, "`country_queries` should contain 3 country names"
assert all(isinstance(name, str) for name in country_queries), "All entries in `country_queries` should be strings"

assert isinstance(country_summaries, list), "`country_summaries` should be a list"
assert len(country_summaries) == 3, "`country_summaries` should contain 3 items"
for item in country_summaries:
    assert isinstance(item, dict), "Each entry in `country_summaries` should be a dictionary"
    for key in ["name", "capital", "population"]:
        assert key in item, f"Each summary should include '{key}'"
    assert isinstance(item["name"], str), "Summary `name` should be text"
    assert isinstance(item["capital"], str), "Summary `capital` should be text"
    assert isinstance(item["population"], int), "Summary `population` should be an integer"

assert isinstance(status_results, dict), "`status_results` should be a dictionary"
for key in ["valid_country", "misspelled_country", "wrong_endpoint"]:
    assert key in status_results, f"`status_results` is missing key: {key}"
    assert isinstance(status_results[key], int), f"`status_results[{key}]` should be an integer status code"
assert status_results["valid_country"] == 200, "The valid country request should return 200"
assert status_results["misspelled_country"] != 200, "Misspelled country should return a non-200 status"
assert status_results["wrong_endpoint"] != 200, "Wrong endpoint should return a non-200 status"

assert isinstance(debug_summary, list), "`debug_summary` should be a list"
assert len(debug_summary) >= 2, "Add at least 2 debugging notes"
assert all(isinstance(note, str) for note in debug_summary), "All `debug_summary` items should be text"

assert isinstance(get_capital, str) and len(get_capital) > 0, "`get_capital` should be a non-empty string"
assert isinstance(post_status, int), "`post_status` should be an integer status code"
assert post_status != 200, "POST to this read endpoint should not return 200"

assert isinstance(extra_field_name, str) and len(extra_field_name) > 0, "`extra_field_name` should be a non-empty string"
assert extra_field_value is not None, "`extra_field_value` should contain data from the response"

notebook3_ready_for_clue = True
print("All checks passed! ✅ You unlocked Clue 3.")


---
## 3.9 Data Detective Challenge — Clue 3

Throughout this course, you'll collect clues in the **Data Detective Challenge**. Each notebook has one clue.
Collect them all to unlock the final certificate in Notebook 6!


In [ ]:
# CHALLENGE: How many people live in the Netherlands according to the API?
# Call the REST Countries API for the Netherlands, extract the population,
# divide by 1,000,000, and round DOWN to a whole number.

import requests

response = requests.get("https://restcountries.com/v3.1/name/netherlands")
data = response.json()

# Write your solution in the next lines:
population = None  # Replace None with the extracted population value
clue_3 = None      # Replace None with int(population / 1_000_000)

print(f"Population: {population}")
print(f"Your answer: {clue_3} million")

# --- Verification ---
if not globals().get("notebook3_ready_for_clue", False):
    print("Complete and pass the checkpoint first to unlock Clue 3.")
elif population is None or clue_3 is None:
    print("Fill in both `population` and `clue_3`, then run this cell again.")
else:
    expected_clue_3 = int(data[0]["population"] / 1_000_000)
    if clue_3 == expected_clue_3:
        print("\n>>> Clue 3 collected!")
        print(f"Write down: clue_3 = {clue_3}")
        print("You'll need this for the final challenge in Notebook 6.")
    else:
        print("Not quite — check how you're extracting `population` from the response.")


---
**Next up:** [Notebook 4 — Working with API Data](4_working_with_api_data.ipynb)